# Machine Learning Workshop

Run the setup cell below once at the start. It downloads the workshop files into this temporary session and sets the working directory.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/2U2N/mnist_workshop.git"
WORKSHOP_DIR = Path("/content/mnist-workshop")

if not WORKSHOP_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(WORKSHOP_DIR)],
        check=True,
    )

required = [
    "data/mnist.pkl.gz",
    "img/Multilayer Perceptron.png",
    "TheNumberThree.png",
]
missing = [name for name in required if not (WORKSHOP_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing workshop files: {missing}. Check the repository folder structure.")

os.chdir(WORKSHOP_DIR)
print("Workshop files ready! You can now run the first workshop code cell under Libraries.")


# Neural Networks
This is a Jupyter Notebook that guides you through training a "simple" neural network. It is based on the book *Neural Networks and Deep Learning* and its acompanying material by Michael Nielsen. You can find the book for free **[here](http://neuralnetworksanddeeplearning.com)** and the open access code **[here](https://github.com/mnielsen/neural-networks-and-deep-learning/tree/master)**
The code is simple for a neural network, but still complex for the uninitiated. That is not problem for you, though. The code will run as it is, as long as it is executed top to bottom. 


## Content
1. Libraries: Load Python libraries to give it capabilities we need
2. Define Python Functions
3. Build the Neural Network
4. Train the Model
5. Inference: Using the Neural Network & the Trained Model to Detect Digits


## Libraries
To start with, we import all **libraries** that we will need. **Libraries** are comparable to add-ons. Many of them come bundled with Python, others can be installed from the web. When an installed **library** is imported, it gives Python abilities that are not part of "base Python". "Base Python" comes without these **libraries** pre-imported to make it as small and as quick as possible.

In [ ]:
import pickle  # Used to save and load Python objects
import gzip  # Compresses files to make them smaller
import numpy as np  # A library that makes working with numbers and mathematical operations easier
import random  # Lets you generate random numbers or make random selections
import time  # Provides tools to track time or create delays in your code
from IPython.display import Image, display # Allows the display of images

## Defining Python Functions

Michael Nielsen, the author of the book that this lesson is based on and of the neural network code, has annotated his functions with a **docstring**, the red text between the """ """. Docstrings are used to provide documentation for the function, explaining what the function does. The docstring is not executed as part of the function's logic, but it serves as a reference for developers (read: us) using or reading the function.

In [ ]:
def load_data():
    """Return the MNIST data as a tuple containing the training data,
    the validation data, and the test data.

    The ``training_data`` is returned as a tuple with two entries.
    The first entry contains the actual training images.  This is a
    numpy ndarray with 50,000 entries.  Each entry is, in turn, a
    numpy ndarray with 784 values, representing the 28 * 28 = 784
    pixels in a single MNIST image.

    The second entry in the ``training_data`` tuple is a numpy ndarray
    containing 50,000 entries.  Those entries are just the digit
    values (0...9) for the corresponding images contained in the first
    entry of the tuple.

    The ``validation_data`` and ``test_data`` are similar, except
    each contains only 10,000 images.

    This is a nice data format, but for use in neural networks it's
    helpful to modify the format of the ``training_data`` a little.
    That's done in the wrapper function ``load_data_wrapper()``, see
    below.
    """
    f = gzip.open('data/mnist.pkl.gz', 'rb')
    u = pickle._Unpickler(f)
    u.encoding = 'latin1'
    training_data, validation_data, test_data = u.load()
    f.close()
    return (training_data, validation_data, test_data)

def load_data_wrapper():
    """Return a tuple containing ``(training_data, validation_data,
    test_data)``. Based on ``load_data``, but the format is more
    convenient for use in our implementation of neural networks.

    In particular, ``training_data`` is a list containing 50,000
    2-tuples ``(x, y)``.  ``x`` is a 784-dimensional numpy.ndarray
    containing the input image.  ``y`` is a 10-dimensional
    numpy.ndarray representing the unit vector corresponding to the
    correct digit for ``x``.

    ``validation_data`` and ``test_data`` are lists containing 10,000
    2-tuples ``(x, y)``.  In each case, ``x`` is a 784-dimensional
    numpy.ndarry containing the input image, and ``y`` is the
    corresponding classification, i.e., the digit values (integers)
    corresponding to ``x``.

    Obviously, this means we're using slightly different formats for
    the training data and the validation / test data.  These formats
    turn out to be the most convenient for use in our neural network
    code."""
    tr_d, va_d, te_d = load_data()
    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [vectorized_result(y) for y in tr_d[1]]
    training_data = list(zip(training_inputs, training_results))
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = list(zip(validation_inputs, va_d[1]))
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = list(zip(test_inputs, te_d[1]))
    return (training_data, validation_data, test_data)

def vectorized_result(j):
    """Return a 10-dimensional unit vector with a 1.0 in the jth
    position and zeroes elsewhere.  This is used to convert a digit
    (0...9) into a corresponding desired output from the neural
    network."""
    e = np.zeros((10, 1))
    e[j] = 1.0
    return e

def sigmoid(z):
    """The sigmoid function."""
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z)*(1-sigmoid(z))

## Build the Neural Network
"Building" is a bit of an overstatement, because dear Michael Nielsen has done all the actual work for us. What remains for us to do is executing the code cell below. Combined, the individual functions will make Python build the neural network. 

Go ahead and execute the code cell below to *build* the neural network.

In [ ]:
class Network(object):

    def __init__(self, sizes):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  The biases and weights for the
        network are initialized randomly, using a Gaussian
        distribution with mean 0, and variance 1.  Note that the first
        layer is assumed to be an input layer, and by convention we
        won't set any biases for those neurons, since biases are only
        ever used in computing the outputs from later layers."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x)
                        for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None):
        """Train the neural network using mini-batch stochastic
        gradient descent.  The ``training_data`` is a list of tuples
        ``(x, y)`` representing the training inputs and the desired
        outputs.  The other non-optional parameters are
        self-explanatory.  If ``test_data`` is provided then the
        network will be evaluated against the test data after each
        epoch, and partial progress printed out.  This is useful for
        tracking progress, but slows things down substantially."""
        if test_data: n_test = len(test_data)
        n = len(training_data)
        for j in range(epochs):
            time1 = time.time()
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            time2 = time.time()
            if test_data:
                print("Epoch {0}: {1} / {2}, took {3:.2f} seconds".format(
                    j, self.evaluate(test_data), n_test, time2-time1))
            else:
                print("Epoch {0} complete in {1:.2f} seconds".format(j, time2-time1))

    def update_mini_batch(self, mini_batch, eta):
        """Update the network's weights and biases by applying
        gradient descent using backpropagation to a single mini batch.
        The ``mini_batch`` is a list of tuples ``(x, y)``, and ``eta``
        is the learning rate."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w-(eta/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b-(eta/len(mini_batch))*nb
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the cost function C_x.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of numpy arrays, similar
        to ``self.biases`` and ``self.weights``."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * \
            sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        # Note that the variable l in the loop below is used a little
        # differently to the notation in Chapter 2 of the book.  Here,
        # l = 1 means the last layer of neurons, l = 2 is the
        # second-last layer, and so on.  It's a renumbering of the
        # scheme in the book, used here to take advantage of the fact
        # that Python can use negative indices in lists.
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        """Return the number of test inputs for which the neural
        network outputs the correct result. Note that the neural
        network's output is assumed to be the index of whichever
        neuron in the final layer has the highest activation."""
        test_results = [(np.argmax(self.feedforward(x)), y)
                        for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activations, y):
        """Return the vector of partial derivatives partial C_x /
        partial a for the output activations."""
        return (output_activations-y)




## Train the Neural Network
Training the neural network also is not that much work for *us*. We can simply have the computer do the heavy lifting. Prerequisite for that is, of course, that we can give it **good training material**. Fortunately for us, the **National Institute of Standards and Technology** of the United States has already aggregated the perfect training material for us: A collection of 70.000 28-by-28 pixel images that each contain one handwritten number and, impotantly, informtion on what number is actually depicted in the image. 

Let's load the data and take a look at it. Note that we are using the function `load_data_wrapper()` that we have defined in the first large code cell above. If you scroll up, and take a look at its  code, you will see that it returns three things: `return (training_data, validation_data, test_data)`. As is common practice within machine learning, the training data is divided into 3 chunks: One large one named `training_data` with 50.000 pairs of images and numbers and two smaller ones with 10.000 pairs each named `validation_data` and `test_data`. 

In [ ]:
training_data , validation_data , test_data = load_data_wrapper()

The division is important to avoid **overfitting** - making the programm extremely good at recognizing the training images but bad at recognizing any number it has not "seen before". In other words, we want the neural network to **generalize**. By training on only a part of all the images, we can afterwards use the other part to check whether the network still generalizes.


### The Training Data
Let's take a look at the data before we proceed to get an understanding of what we are actually dealing with here.
Below is the code for visualizing a few, randomly selected images from the training data. I have annoteded the code with #comments to explain what is going on.


In [ ]:
import matplotlib.pyplot as plt #import matplotlib, a common visualization library
import random   #import the library random, that allows python to generate random numbers


num_images = 5  # Decide how many images to visualize, you can change this parameter if you'd like
random_indices = random.sample(range(len(training_data)), num_images)   # generate a random number between 1 and 50,000 per image that is supposed to be visualized

plt.figure(figsize=(10, 2)) # initiate the visualization by deciding how big the visualization should be: width 10, height 2

for i, idx in enumerate(random_indices): #for every random number generated above select the corresponding image in the list of 50,000 images

    image_data = training_data[idx][0].reshape(28, 28) # in the training data the images are stored as lists of values between black (0) and white (1), this step reshapes them into a grid of pixels that we can look at 
    
    label = np.argmax(training_data[idx][1])  # This gets the corresponding label for the image
    
    # This defines how the image is actually supposed to be visualized
    plt.subplot(1, num_images, i + 1) # defines a location for each of the 5 images
    plt.imshow(image_data, cmap='gray') # defines which of the 5 images is put into that location
    plt.title(f'Label: {label}')  # Add the corresponding label
    plt.axis('off')  # Hide the axes for clarity

plt.show()

### An Individual Image

Let's take a closer look at an individual image. For example, image number 26869 from our 50,000 training images. 

In [ ]:
# Reshape the list of pixel values into the 28-by-28 pixel grid of the image
image_data = training_data[26869][0].reshape(28, 28)

# Get the corresponding label
label = np.argmax(training_data[26869][1]) 

 # initiate the visualization by deciding how big it should be: width 10, height 10
fig, ax = plt.subplots(figsize=(10, 10))

# Create the image in the computers working memory (without printing it yet)
cax = ax.imshow(image_data, cmap='gray')

# Loop over data dimensions and plop the value of each pixel (0=black and 1=white) into the position of each pixel 
for i in range(28):
    for j in range(28):
        pixel_value = image_data[i, j]
        text = ax.text(j, i, f"{pixel_value:.2f}".lstrip('0'),
                       ha="center", va="center", color="red", fontsize=10)

ax.set_title(f'Label: {label}') # determine titel
ax.axis('off')  # disable printing of axes

# print the image
plt.tight_layout() 
plt.show()

### The Output 
What we want to get out of the neural network is the answer to "Which number is depicted in the image?". To enable the neural network to give us this answer, we give it **10 output neurons** and associate each neuron with one digit 0, 1, 2, 3, 4, 5, 6, 7, 8, or 9. This way, the neural network can show that it has identified a **2** by sending activation only to the neuron that is associated with that digit. 

Conveniently the MNIST data that we are training comes already prepared. Take a look at the label, i.e. the correct answer answer to the image above, by executing the next line. 

In [ ]:
training_data[26869][1]

Does it make sense to you that this is the ideal response?

Does it make more sense, if you consider it in the context of the image below?


In [ ]:
# Displaying a Screenshot from the 3Blueonebrown YouTube video
display(Image(filename='img/Multilayer Perceptron.png', width=700))

## The Neural Network
Now we recreate the neural network as it is depicted in the screenshot above. 

We create a new object called ``net`` by giving a list of numbers ``[784, 16, 16, 10]`` to the function ``Network``. The ``Network`` function will create a layer in the neural network ``net`` for each item in the list. The value of each item determines the number of neurons per layer.

Thus, by giving it the list ``[784, 16, 16, 10]`` we create an input layer with 784 neurons, two middle layers, called *hidden layers* with 16 neurons each, and an output layer with 10 neurons - one for each digit. 

### Training 

To train our neural network, we apply the training methods "SGD" (Stochastic Gradient Descent) as defined above, to our network. We provide the method with the following parameters:

- training data <- combination of 50,000 **images and (correct) labels**
- 10 <- number of **epochs**, is a training run through all training images
- 10 <- **batch-size**, the number of images that are processed before the parameters are changed.
- 3.0 <- **learning rate** describes how big each change in parameters should be.

In [ ]:
# Creating the Neural Network
net = Network([784, 16, 16, 10])

# Initiating the training 
net.SGD(training_data,
        10,   # epochs
        10,   # batch-size
        3,    # learning rate
        test_data=test_data)

### How to interpret the output of the code cell above:
- Epoch Number: Number of finished training runs. Each epoch involves computing the neural network for each image-label combination
- xxxxx / 10000: After each epoch the performance of the neural network is evaluated with the images from the ``test_data`` that the neural network has not seen during training. The number indicates how many of the 10,000 test-images it recognizes correctly.
- time: time requirement per epoch

### Can you improve the Performance?

- try running it with more or fewer hidden layers
- try running it with a different learning rate 
- do the changes improve or worsen the result?
- how many epochs does the network need each time to reach its best test-score?


## Inference
Inference is the fancy term for *using a trained network* and it is the whole reason why neural networks are trained in the first place. 
Below is a function, written by ChatGPT, that takes the trained network and an "image" and returns the **detected number** and **values of the neurons in the output layer**.

**Tasks:**
1. Take a picture of a number written in dark color on bright background
 - you can for example use [kleki.com](https://kleki.com) to draw numbers 
3. Convert the image to png or jpeg (for example by taking a screenshot of it)
4. Upload the new image to Jupyter via *drag and drop* to the directory on the left.
5. Replace the "image_path_here" with the name of the image 
6. Execute the code cell below to load the functions



In [ ]:
image_path = "TheNumberThree.png"

In [ ]:
from PIL import Image, ImageOps

def predict(network, image):
    """
    Perform inference using the trained neural network.

    Parameters:
    - network: An instance of the trained Network class.
    - image: A NumPy array representing the input image. 
             It should be preprocessed to match the input layer's size.

    Returns:
    - prediction: The predicted class label (integer).
    """
    # Ensure the image is a NumPy array
    if not isinstance(image, np.ndarray):
        raise TypeError("Image must be a NumPy array.")
    
    # Flatten the image if it's not already a 1D array
    if image.ndim > 1:
        image = image.flatten()
    
    # Reshape the image to a column vector (input_size x 1)
    image = image.reshape(-1, 1)
    
    # Normalize the image data if required (e.g., [0, 1] range)
    # Uncomment and adjust the following line if normalization was used during training
    # image = image / 255.0
    
    # Perform feedforward to get the output activations
    output_activations = network.feedforward(image)
    
    # Determine the predicted label by finding the index with the highest activation
    prediction = np.argmax(output_activations)

    rounded_output_activations = np.round(output_activations, 2)
    
    # Print the prediction
    print(f"The neural network predicts the image to show a {prediction}")

    return rounded_output_activations


def image_to_array (image_path):
    image = Image.open(image_path)

    image_width, image_height = image.size

    square_size = min(image_width, image_height)
    
    # Calculate the coordinates for the crop box
    left = (image_width - square_size) // 2
    top = (image_height - square_size) // 2
    right = left + square_size
    bottom = top + square_size
    
    # Crop the image
    cropped_image = image.crop((left, top, right, bottom))

    # Convert the image to grayscale
    gray_image = cropped_image.convert("L")

    small_image = gray_image.resize((28, 28))

    adjusted_image = ImageOps.invert(ImageOps.autocontrast(small_image))
    
    # Convert the grayscale image to a numpy array
    image_array = np.array(adjusted_image)

    # Normalize the pixel values to be between 0 (brightest) and 1 (darkest)
    brightness_array = image_array / 255.0


    brightness_array_flat = brightness_array.flatten()
        
    return(brightness_array_flat, cropped_image, gray_image, small_image, adjusted_image)


In [ ]:
flat_array, cropped_image, gray_image, small_image, adjusted_image = image_to_array(image_path)

In [ ]:
def visualize_images(cropped_image, gray_image, small_image, adjusted_image):
    # Create a figure with 2x2 grid for the images
    fig, axs = plt.subplots(2, 2, figsize=(8, 8))

    # Plot the cropped image
    axs[0, 0].imshow(cropped_image)
    axs[0, 0].set_title("Cropped Image")
    axs[0, 0].axis('off')  # Hide axis

    # Plot the grayscale image
    axs[0, 1].imshow(gray_image, cmap='gray')
    axs[0, 1].set_title("Grayscale Image")
    axs[0, 1].axis('off')

    # Plot the small (28x28) image
    axs[1, 0].imshow(small_image, cmap='gray')
    axs[1, 0].set_title("Small Image (28x28)")
    axs[1, 0].axis('off')

    # Plot the adjusted image
    axs[1, 1].imshow(adjusted_image, cmap='gray')
    axs[1, 1].set_title("Adjusted Image (Autocontrast)")
    axs[1, 1].axis('off')

    # Adjust layout for better spacing
    plt.tight_layout()
    plt.show()

# Call the function to visualize the images
visualize_images(cropped_image, gray_image, small_image, adjusted_image)


In [ ]:
# Reshape the list of pixel values into the 28-by-28 pixel grid of the image
image_data = flat_array.reshape(28, 28)
 

 # initiate the visualization by deciding how big it should be: width 10, height 10
fig, ax = plt.subplots(figsize=(10, 10))

# Create the image (without printing it yet)
cax = ax.imshow(image_data, cmap='gray')

# Loop over data dimensions and plop the value of each pixel (0=black and 1=white) into the position of each pixel 
for i in range(28):
    for j in range(28):
        pixel_value = image_data[i, j]
        text = ax.text(j, i, f"{pixel_value:.2f}".lstrip('0'),
                       ha="center", va="center", color="red", fontsize=10)

ax.axis('off')  # disable printing of axes

# print the image
plt.tight_layout() 
plt.show()

In [ ]:
predict(net, flat_array)